# Training a Block-Diffusion LLM — Getting Started

A hands-on first run of the `train/` stack: take a pretrained **autoregressive** HF checkpoint and
SFT it into a **block-diffusion** LLM — one set of weights you can later serve three ways
(`causal`, `diffusion`, `self-spec`) with the `inference/` stack.

The trick is a two-part objective on the *same* weights:

- a **block-diffusion** loss on a *noised* view of each response (predict masked tokens in a block, in parallel), and
- an **AR auxiliary** loss on a *clean* view (ordinary next-token), so the causal head keeps working.

Combined as `loss = (diff + w·ar) / (1 + w)`. That's what lets one checkpoint decode either way.

| model family | trainer | objective layout |
|---|---|---|
| dense (Qwen3) | `HFBlockDiffusion` | `[S | x_t | x_0]` per row |
| hybrid GDA (**Qwen3.5**) | `HFBlockDiffusionHybrid` | FLARE two-stream `[x0 ; xt_1 … xt_V]` |

This notebook writes a tiny toy dataset, runs a 2-step smoke on **one GPU**, inspects the resulting
checkpoint, then shows the real multi-GPU command. See [`README.md`](README.md) for the full map.

## Prerequisites

1. **A GPU** (the smoke needs one) and the training deps installed — `torch` (FSDP2), `transformers`,
   `triton` + `fla` for the hybrid gated-delta kernels, and `datasets`. From the repo root:
   `pip install -r train/requirements.txt` (or your project's env).
2. **Run this notebook from the `train/` directory** so the relative paths resolve. Training itself is
   launched through the **top-level `train.py` shim** (one directory up) — never `train/train.py`
   directly (that shadows the package on `sys.path`).
3. **A base checkpoint** — an HF repo id or local path to an autoregressive causal LM (e.g.
   `Qwen/Qwen3.5-4B`). The trainer adds a `<|mask|>` token and grows the embedding by one row.
4. **Data** — chat-format JSONL (below). The default `--dataset` is an HF id, but a **local directory**
   of `*.jsonl` files works too; we use a toy one here.

## Configure

Point at your base model and pick where the smoke writes. If you only want to *read* the guide without
running anything, leave `RUN_SMOKE = False`.

In [ ]:
import os, pathlib, subprocess, sys

# --- base checkpoint to convert (HF repo id or local path) ---
MODEL = os.environ.get('TRIDA_BASE_MODEL', 'Qwen/Qwen3.5-4B')

# --- run the tiny 1-GPU smoke in this notebook? (needs a GPU + deps) ---
RUN_SMOKE = os.environ.get('TRIDA_RUN_SMOKE', '0') == '1'

TRAIN = pathlib.Path.cwd()
REPO  = TRAIN.parent                      # train.py shim lives at the repo root
assert (TRAIN / 'train.py').exists() and (TRAIN / 'hf_block_diffusion.py').exists(), \
    'Run this notebook from the train/ directory.'
assert (REPO / 'train.py').exists(), 'Expected the top-level train.py shim at the repo root.'

TOY_DIR = TRAIN / 'docs' / '_toy_data'    # tiny dataset we create below
OUT_DIR = REPO / 'checkpoints' / 'tutorial_smoke'
print('base model :', MODEL)
print('repo root  :', REPO)
print('run smoke  :', RUN_SMOKE)

## 1. The data format

Each JSONL line is one conversation: a `messages` list (OpenAI chat shape) and an optional top-level
`tools` schema. The renderer applies the tokenizer's chat template, supervises the **assistant** turns
(everything else is masked to `-100`), and length-buckets the rows.

```json
{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

`--multiturn` supervises *every* assistant turn; with `--pack` whole conversations are bin-packed into
each row (document-isolated: recurrence, attention, and RoPE all reset per document). Let's write a
handful of toy examples:

In [ ]:
import json
TOY_DIR.mkdir(parents=True, exist_ok=True)
toy = [
    {'messages': [{'role': 'user', 'content': 'What is 2 + 2?'},
                  {'role': 'assistant', 'content': 'It is 4.'}]},
    {'messages': [{'role': 'user', 'content': 'Name a primary color.'},
                  {'role': 'assistant', 'content': 'Red is a primary color.'},
                  {'role': 'user', 'content': 'Another?'},
                  {'role': 'assistant', 'content': 'Blue is another.'}]},
    {'messages': [{'role': 'user', 'content': 'Say hello in French.'},
                  {'role': 'assistant', 'content': 'Bonjour!'}]},
]
with open(TOY_DIR / 'toy.jsonl', 'w') as f:
    for r in toy * 16:            # repeat so a couple of packed steps have data
        f.write(json.dumps(r) + '\n')
print('wrote', TOY_DIR / 'toy.jsonl', '(', len(toy)*16, 'lines )')

## 2. A minimal training run (1 GPU, 2 steps)

The same entry point as a real run, just tiny: `torchrun` with one process, `bd_size 8`, short
sequences, `--max_steps 2`. It launches the **top-level `train.py`** from the repo root. `--fused_ce`
needs Liger; drop it if unavailable. For a hybrid model without the compiled kernels, add
`--allow_slow_linear` (slow, but runs).

> This downloads the base model on first run and needs a GPU. Set `RUN_SMOKE = True` (or
> `TRIDA_RUN_SMOKE=1`) in **Configure** to actually execute it.

In [ ]:
cmd = [
    'torchrun', '--nnodes=1', '--nproc_per_node=1', 'train.py',
    '--model_id', MODEL,
    '--dataset', str(TOY_DIR),
    '--pack', '--multiturn', '--keep_all_reasoning',
    '--linear_block_mode', 'bidirectional',
    '--bd_size', '8', '--ar_loss_weight', '0.1', '--fused_ce',
    '--max_length', '2048', '--max_response_length', '2048',
    '--length_buckets', '1024,2048', '--response_buckets', '1024,2048',
    '--pack_examples', '4', '--micro_batch_size', '1', '--grad_accum', '1',
    '--max_steps', '2', '--warmup', '1', '--lr', '1e-5',
    '--save_dir', str(OUT_DIR), '--save_every', '2', '--save_optim_state', '--log_every', '1',
]
print('$', ' '.join(cmd), '\n  (cwd =', REPO, ')\n')
if RUN_SMOKE:
    subprocess.run(cmd, cwd=REPO, check=True)
else:
    print('RUN_SMOKE is False — command shown only. Set it True in Configure to run.')

## 3. Inspect the checkpoint

A saved dir is a normal `from_pretrained`-loadable HF model plus two extras: the tokenizer already has
the `<|mask|>` token, and `block_diffusion.json` records `bd_size` / `mask_id` / `ar_loss_weight`.
With `--save_optim_state`, `train_state/` + `train_state.json` make it exactly resumable.

In [ ]:
import json, pathlib
ck = OUT_DIR / 'step_2'
if ck.exists():
    print('contents:', sorted(p.name for p in ck.iterdir()))
    print('block_diffusion.json:', json.load(open(ck / 'block_diffusion.json')))
else:
    print('No checkpoint yet — run the smoke above first (RUN_SMOKE=True).')

## 4. Scaling to a real run

The production command — 8 GPUs, packed multi-turn, the flagship Qwen3.5-4B FLARE recipe:

```bash
# from the repo root
torchrun --nnodes=1 --nproc_per_node=8 train.py \
  --model_id Qwen/Qwen3.5-4B \
  --dataset <hf-dataset-or-local-jsonl-dir> \
  --pack --multiturn --keep_all_reasoning \
  --linear_block_mode bidirectional \
  --bd_size 32 --ar_loss_weight 0.1 --fused_ce --compile_mlp \
  --max_length 32768 --length_buckets 2048,4096,8192,16384,20480,24576,28672,32768 \
  --micro_batch_size 1 --grad_accum 2 \
  --save_dir checkpoints/<run> --save_every 1000 --save_optim_state
```

Multi-node: run one `torchrun` per node with `--nnodes=N --node_rank=<i>` and a shared
`--master_addr/--master_port` (see the `slurm/` sbatches for a working 2-node launcher).

### Flags that matter

| flag | what it does |
|---|---|
| `--bd_size` | block length (32 = the trained geometry; smaller = closer to AR) |
| `--ar_loss_weight w` | AR aux weight in `(diff + w·ar)/(1+w)` |
| `--loss_weighting` | `weighted` = 1/γ ELBO weighting of the diffusion CE; `uniform` = mean |
| `--linear_block_mode` | hybrid only: `bidirectional` (real method) vs `causal` (legacy approx) |
| `--pack --multiturn` | packed whole conversations (document-isolated) |
| `--fused_ce`, `--compile_mlp` | throughput (~1.7× supervised tok/s/GPU); omit `--activation_offload` unless memory-bound |
| `--save_optim_state` | exactly-resumable checkpoints |
| `--resume <ckpt>` | resume weights (+ optimizer/step if present) — pair with `--model_id <ckpt>` |

Two research recipes also live here: `--pos_decay_gamma G` (DFlash intra-block positional loss decay)
and `--full_mask` (100%-mask single-view). See `README.md` for what each is for.

## 5. Serve what you trained

A checkpoint from here is directly servable by the `inference/` stack. Open
[`../inference/getting_started.ipynb`](../inference/getting_started.ipynb), point `MODEL` at your
`checkpoints/<run>/step_<N>` dir, and pick a decode mode (`causal` / `diffusion` / `self-spec`).
SGLang needs a one-time checkpoint-shaping step; vLLM serves the raw dir. See
[`../inference/MODEL_CONTRACT.md`](../inference/MODEL_CONTRACT.md) for compatibility.

## Next steps

- **[`README.md`](README.md)** — the full `train/` map: model routing, the two-stream path, the data
  pipeline, and the vendored gated-delta kernels (PolyForm Noncommercial — read before commercial use).
- **Root `README.md`** — the block-diffusion concept end to end.
- **`slurm/`** — production launchers, including the multi-node torchrun rendezvous.
- Scale up `--dataset`, raise `--max_steps`, and watch `diff_loss` / `ar_loss` (add `--wandb`).